In [1]:
import pandas as pd
import numpy as np

# Load raw datasets
sales = pd.read_csv("../data/raw/sales_data-2.csv")
inventory = pd.read_csv("../data/raw/inventory_data.csv")
suppliers = pd.read_csv("../data/raw/suppliers_data.csv")
purchase_orders = pd.read_csv("../data/raw/purchase_orders_data.csv")

print("===== DATASET SHAPES =====")
print("Sales:", sales.shape)
print("Inventory:", inventory.shape)
print("Suppliers:", suppliers.shape)
print("Purchase Orders:", purchase_orders.shape)

print("\n===== SALES =====")
display(sales.head())

print("\n===== INVENTORY =====")
display(inventory.head())

print("\n===== SUPPLIERS =====")
display(suppliers.head())

print("\n===== PURCHASE ORDERS =====")
display(purchase_orders.head())

===== DATASET SHAPES =====
Sales: (5000, 6)
Inventory: (200, 6)
Suppliers: (50, 5)
Purchase Orders: (1000, 6)

===== SALES =====


,Sale_ID,Product_ID,Store_ID,Sale_Date,Quantity_Sold,Revenue
0,0d12b607,P001,S102,2024-01-01,322,2556
1,2e13fe47,P003,S101,2024-01-02,28,42606
2,c05afd2d,P003,S104,2024-01-03,292,19551
3,aff62361,P004,S101,2024-01-04,280,26625
4,9d190af1,P003,S101,2024-01-05,353,24001



===== INVENTORY =====


,Product_ID,Store_ID,Warehouse_ID,Stock_Level,Reorder_Level,Last_Updated
0,P005,S101,W003,498,163,2024-02-01
1,P004,S103,W003,24,178,2024-02-02
2,P002,S101,W002,245,188,2024-02-03
3,P003,S101,W003,61,148,2024-02-04
4,P002,S101,W002,57,155,2024-02-05



===== SUPPLIERS =====


,Supplier_ID,Supplier_Name,Product_ID,Lead_Time (days),Order_Frequency
0,SUP001,Murray-Ramirez,P002,9,Weekly
1,SUP002,"Thomas, Meyer and Campbell",P005,9,Weekly
2,SUP003,Acosta Inc,P005,4,Monthly
3,SUP004,"Evans, Brown and Turner",P002,8,Weekly
4,SUP005,"Johnson, Wise and Barr",P003,6,Biweekly



===== PURCHASE ORDERS =====


,Order_ID,Product_ID,Supplier_ID,Order_Date,Quantity,Arrival_Date
0,PO0001,P001,SUP002,2024-01-01,158,2024-01-06
1,PO0002,P005,SUP019,2024-01-02,440,2024-01-07
2,PO0003,P002,SUP036,2024-01-03,423,2024-01-08
3,PO0004,P004,SUP021,2024-01-04,88,2024-01-09
4,PO0005,P001,SUP002,2024-01-05,329,2024-01-10


In [3]:
datasets = {
    "Sales": sales,
    "Inventory": inventory,
    "Suppliers": suppliers,
    "Purchase Orders": purchase_orders
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"{name.upper()} DATA QUALITY CHECK")
    print("=" * 60)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)

# Duplicate ID checks
print("\n" + "=" * 60)
print("DUPLICATE ID CHECK")
print("=" * 60)

print("Duplicate Sale_IDs:",
      sales["Sale_ID"].duplicated().sum())

print("Duplicate Inventory Product-Store combinations:",
      inventory.duplicated(subset=["Product_ID", "Store_ID"]).sum())

print("Duplicate Supplier_IDs:",
      suppliers["Supplier_ID"].duplicated().sum())

print("Duplicate Order_IDs:",
      purchase_orders["Order_ID"].duplicated().sum())


SALES DATA QUALITY CHECK

Missing Values:
Sale_ID          0
Product_ID       0
Store_ID         0
Sale_Date        0
Quantity_Sold    0
Revenue          0
dtype: int64

Duplicate Rows:
0

Data Types:
Sale_ID            str
Product_ID         str
Store_ID           str
Sale_Date          str
Quantity_Sold    int64
Revenue          int64
dtype: object

INVENTORY DATA QUALITY CHECK

Missing Values:
Product_ID       0
Store_ID         0
Warehouse_ID     0
Stock_Level      0
Reorder_Level    0
Last_Updated     0
dtype: int64

Duplicate Rows:
0

Data Types:
Product_ID         str
Store_ID           str
Warehouse_ID       str
Stock_Level      int64
Reorder_Level    int64
Last_Updated       str
dtype: object

SUPPLIERS DATA QUALITY CHECK

Missing Values:
Supplier_ID         0
Supplier_Name       0
Product_ID          0
Lead_Time (days)    0
Order_Frequency     0
dtype: int64

Duplicate Rows:
0

Data Types:
Supplier_ID           str
Supplier_Name         str
Product_ID            str
Lead_Tim

In [4]:
# ==========================================
# BLOCK 3: INVENTORY GRANULARITY CHECK
# ==========================================

print("===== INVENTORY RECORD DISTRIBUTION =====")

print("\nTotal inventory rows:", len(inventory))

print("\nUnique Product-Store combinations:",
      inventory[["Product_ID", "Store_ID"]].drop_duplicates().shape[0])

print("\nProduct-Store combinations appearing more than once:")
product_store_counts = (
    inventory
    .groupby(["Product_ID", "Store_ID"])
    .size()
    .reset_index(name="Record_Count")
    .sort_values("Record_Count", ascending=False)
)

display(product_store_counts.head(15))

print("\nExact duplicate Product + Store + Date records:",
      inventory.duplicated(
          subset=["Product_ID", "Store_ID", "Last_Updated"]
      ).sum())

print("\nInventory date range:")
print("First:", inventory["Last_Updated"].min())
print("Last :", inventory["Last_Updated"].max())

===== INVENTORY RECORD DISTRIBUTION =====

Total inventory rows: 200

Unique Product-Store combinations: 20

Product-Store combinations appearing more than once:


,Product_ID,Store_ID,Record_Count
8,P003,S101,18
2,P001,S103,15
4,P002,S101,15
9,P003,S102,14
13,P004,S102,12
16,P005,S101,11
14,P004,S103,11
18,P005,S103,11
5,P002,S102,11
11,P003,S104,10



Exact duplicate Product + Store + Date records: 0

Inventory date range:
First: 2024-02-01
Last : 2024-08-18


In [5]:
# ==========================================
# BLOCK 4: DATE & CATEGORY AUDIT
# ==========================================

print("=" * 60)
print("DATE RANGE ANALYSIS")
print("=" * 60)

print("\nSales:")
print("First Sale Date:", sales["Sale_Date"].min())
print("Last Sale Date :", sales["Sale_Date"].max())

print("\nInventory:")
print("First Updated Date:", inventory["Last_Updated"].min())
print("Last Updated Date :", inventory["Last_Updated"].max())

print("\nPurchase Orders:")
print("First Order Date:", purchase_orders["Order_Date"].min())
print("Last Order Date :", purchase_orders["Order_Date"].max())

print("\nFirst Arrival Date:", purchase_orders["Arrival_Date"].min())
print("Last Arrival Date :", purchase_orders["Arrival_Date"].max())


print("\n" + "=" * 60)
print("CATEGORICAL VALUE ANALYSIS")
print("=" * 60)

print("\nSales - Products:")
print(sorted(sales["Product_ID"].unique()))

print("\nSales - Stores:")
print(sorted(sales["Store_ID"].unique()))

print("\nInventory - Warehouses:")
print(sorted(inventory["Warehouse_ID"].unique()))

print("\nSupplier - Order Frequency:")
print(suppliers["Order_Frequency"].value_counts())

print("\nSupplier - Products:")
print(sorted(suppliers["Product_ID"].unique()))

print("\nPurchase Orders - Products:")
print(sorted(purchase_orders["Product_ID"].unique()))

print("\nPurchase Orders - Suppliers:")
print("Unique suppliers in purchase orders:",
      purchase_orders["Supplier_ID"].nunique())

DATE RANGE ANALYSIS

Sales:
First Sale Date: 2024-01-01
Last Sale Date : 2037-09-08

Inventory:
First Updated Date: 2024-02-01
Last Updated Date : 2024-08-18

Purchase Orders:
First Order Date: 2024-01-01
Last Order Date : 2026-09-26

First Arrival Date: 2024-01-06
Last Arrival Date : 2026-10-01

CATEGORICAL VALUE ANALYSIS

Sales - Products:
['P001', 'P002', 'P003', 'P004', 'P005']

Sales - Stores:
['S101', 'S102', 'S103', 'S104']

Inventory - Warehouses:
['W001', 'W002', 'W003']

Supplier - Order Frequency:
Order_Frequency
Weekly      19
Biweekly    17
Monthly     14
Name: count, dtype: int64

Supplier - Products:
['P001', 'P002', 'P003', 'P004', 'P005']

Purchase Orders - Products:
['P001', 'P002', 'P003', 'P004', 'P005']

Purchase Orders - Suppliers:
Unique suppliers in purchase orders: 50


In [6]:
# ==========================================
# BLOCK 5: SALES DATE DISTRIBUTION AUDIT
# ==========================================

sales_dates = pd.to_datetime(sales["Sale_Date"])

print("=" * 60)
print("SALES DATE DISTRIBUTION")
print("=" * 60)

print("\nEarliest date:", sales_dates.min())
print("Latest date :", sales_dates.max())

print("\nNumber of unique sale dates:",
      sales_dates.nunique())

print("\nNumber of sales records by year:")
print(sales_dates.dt.year.value_counts().sort_index())

print("\nNumber of sales records by month:")
monthly_sales_records = (
    sales_dates
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

print(monthly_sales_records)

print("\nSales records from 2024:")
print((sales_dates.dt.year == 2024).sum())

print("\nSales records after 2024:")
print((sales_dates.dt.year > 2024).sum())

SALES DATE DISTRIBUTION

Earliest date: 2024-01-01 00:00:00
Latest date : 2037-09-08 00:00:00

Number of unique sale dates: 5000

Number of sales records by year:
Sale_Date
2024    366
2025    365
2026    365
2027    365
2028    366
2029    365
2030    365
2031    365
2032    366
2033    365
2034    365
2035    365
2036    366
2037    251
Name: count, dtype: int64

Number of sales records by month:
Sale_Date
2024-01    31
2024-02    29
2024-03    31
2024-04    30
2024-05    31
           ..
2037-05    31
2037-06    30
2037-07    31
2037-08    31
2037-09     8
Freq: M, Name: count, Length: 165, dtype: int64

Sales records from 2024:
366

Sales records after 2024:
4634


In [7]:
# ==========================================
# BLOCK 6: NUMERIC DATA VALIDATION
# ==========================================

print("=" * 60)
print("NUMERIC DATA VALIDATION")
print("=" * 60)

print("\nSALES")
print("-" * 40)
print("Quantity Sold:")
print("  Minimum:", sales["Quantity_Sold"].min())
print("  Maximum:", sales["Quantity_Sold"].max())
print("  Mean   :", round(sales["Quantity_Sold"].mean(), 2))

print("\nRevenue:")
print("  Minimum:", sales["Revenue"].min())
print("  Maximum:", sales["Revenue"].max())
print("  Mean   :", round(sales["Revenue"].mean(), 2))

print("\nNegative Quantity:", (sales["Quantity_Sold"] < 0).sum())
print("Zero Quantity:", (sales["Quantity_Sold"] == 0).sum())

print("\nNegative Revenue:", (sales["Revenue"] < 0).sum())
print("Zero Revenue:", (sales["Revenue"] == 0).sum())


print("\nINVENTORY")
print("-" * 40)
print("Stock Level:")
print("  Minimum:", inventory["Stock_Level"].min())
print("  Maximum:", inventory["Stock_Level"].max())
print("  Mean   :", round(inventory["Stock_Level"].mean(), 2))

print("\nReorder Level:")
print("  Minimum:", inventory["Reorder_Level"].min())
print("  Maximum:", inventory["Reorder_Level"].max())
print("  Mean   :", round(inventory["Reorder_Level"].mean(), 2))

print("\nNegative Stock:", (inventory["Stock_Level"] < 0).sum())
print("Negative Reorder Level:", (inventory["Reorder_Level"] < 0).sum())


print("\nSUPPLIERS")
print("-" * 40)
print("Lead Time:")
print("  Minimum:", suppliers["Lead_Time (days)"].min())
print("  Maximum:", suppliers["Lead_Time (days)"].max())
print("  Mean   :", round(suppliers["Lead_Time (days)"].mean(), 2))

print("\nOrder Frequency:")
print(suppliers["Order_Frequency"].value_counts())


print("\nPURCHASE ORDERS")
print("-" * 40)
print("Quantity:")
print("  Minimum:", purchase_orders["Quantity"].min())
print("  Maximum:", purchase_orders["Quantity"].max())
print("  Mean   :", round(purchase_orders["Quantity"].mean(), 2))

print("\nNegative Quantity:", (purchase_orders["Quantity"] < 0).sum())
print("Zero Quantity:", (purchase_orders["Quantity"] == 0).sum())


# Calculate actual order-to-arrival duration
purchase_orders["Order_Date"] = pd.to_datetime(purchase_orders["Order_Date"])
purchase_orders["Arrival_Date"] = pd.to_datetime(purchase_orders["Arrival_Date"])

purchase_orders["Actual_Lead_Time"] = (
    purchase_orders["Arrival_Date"] - purchase_orders["Order_Date"]
).dt.days

print("\nActual Order-to-Arrival Time:")
print("  Minimum:", purchase_orders["Actual_Lead_Time"].min(), "days")
print("  Maximum:", purchase_orders["Actual_Lead_Time"].max(), "days")
print("  Mean   :", round(purchase_orders["Actual_Lead_Time"].mean(), 2), "days")

print("\nPurchase orders with negative duration:",
      (purchase_orders["Actual_Lead_Time"] < 0).sum())

NUMERIC DATA VALIDATION

SALES
----------------------------------------
Quantity Sold:
  Minimum: 10
  Maximum: 499
  Mean   : 258.42

Revenue:
  Minimum: 1003
  Maximum: 49973
  Mean   : 25174.0

Negative Quantity: 0
Zero Quantity: 0

Negative Revenue: 0
Zero Revenue: 0

INVENTORY
----------------------------------------
Stock Level:
  Minimum: 11
  Maximum: 498
  Mean   : 224.96

Reorder Level:
  Minimum: 50
  Maximum: 198
  Mean   : 124.79

Negative Stock: 0
Negative Reorder Level: 0

SUPPLIERS
----------------------------------------
Lead Time:
  Minimum: 3
  Maximum: 9
  Mean   : 6.22

Order Frequency:
Order_Frequency
Weekly      19
Biweekly    17
Monthly     14
Name: count, dtype: int64

PURCHASE ORDERS
----------------------------------------
Quantity:
  Minimum: 50
  Maximum: 499
  Mean   : 276.49

Negative Quantity: 0
Zero Quantity: 0

Actual Order-to-Arrival Time:
  Minimum: 5 days
  Maximum: 5 days
  Mean   : 5.0 days

Purchase orders with negative duration: 0


In [8]:
print("=" * 60)
print("MISSING NUMERIC VALIDATION")
print("=" * 60)

print("\nINVENTORY")
print("Stock Level:")
print("  Minimum:", inventory["Stock_Level"].min())
print("  Maximum:", inventory["Stock_Level"].max())
print("  Mean   :", round(inventory["Stock_Level"].mean(), 2))

print("\nReorder Level:")
print("  Minimum:", inventory["Reorder_Level"].min())
print("  Maximum:", inventory["Reorder_Level"].max())
print("  Mean   :", round(inventory["Reorder_Level"].mean(), 2))

print("\nNegative Stock:", (inventory["Stock_Level"] < 0).sum())
print("Negative Reorder Level:", (inventory["Reorder_Level"] < 0).sum())


print("\nSUPPLIERS")
print("Lead Time:")
print("  Minimum:", suppliers["Lead_Time (days)"].min())
print("  Maximum:", suppliers["Lead_Time (days)"].max())
print("  Mean   :", round(suppliers["Lead_Time (days)"].mean(), 2))

print("\nOrder Frequency:")
print(suppliers["Order_Frequency"].value_counts())

MISSING NUMERIC VALIDATION

INVENTORY
Stock Level:
  Minimum: 11
  Maximum: 498
  Mean   : 224.96

Reorder Level:
  Minimum: 50
  Maximum: 198
  Mean   : 124.79

Negative Stock: 0
Negative Reorder Level: 0

SUPPLIERS
Lead Time:
  Minimum: 3
  Maximum: 9
  Mean   : 6.22

Order Frequency:
Order_Frequency
Weekly      19
Biweekly    17
Monthly     14
Name: count, dtype: int64


In [9]:
# ==========================================
# BLOCK 7: ETL CLEANING & STANDARDIZATION
# ==========================================

# Create clean copies
sales_clean = sales.copy()
inventory_clean = inventory.copy()
suppliers_clean = suppliers.copy()
purchase_orders_clean = purchase_orders.copy()


# ------------------------------------------
# 1. Standardize date columns
# ------------------------------------------

sales_clean["Sale_Date"] = pd.to_datetime(
    sales_clean["Sale_Date"],
    errors="coerce"
)

inventory_clean["Last_Updated"] = pd.to_datetime(
    inventory_clean["Last_Updated"],
    errors="coerce"
)

purchase_orders_clean["Order_Date"] = pd.to_datetime(
    purchase_orders_clean["Order_Date"],
    errors="coerce"
)

purchase_orders_clean["Arrival_Date"] = pd.to_datetime(
    purchase_orders_clean["Arrival_Date"],
    errors="coerce"
)


# ------------------------------------------
# 2. Standardize text columns
# ------------------------------------------

text_columns = {
    "sales": ["Sale_ID", "Product_ID", "Store_ID"],
    "inventory": ["Product_ID", "Store_ID", "Warehouse_ID"],
    "suppliers": ["Supplier_ID", "Supplier_Name", "Product_ID", "Order_Frequency"],
    "purchase_orders": ["Order_ID", "Product_ID", "Supplier_ID"]
}

for column in text_columns["sales"]:
    sales_clean[column] = sales_clean[column].astype(str).str.strip()

for column in text_columns["inventory"]:
    inventory_clean[column] = inventory_clean[column].astype(str).str.strip()

for column in text_columns["suppliers"]:
    suppliers_clean[column] = suppliers_clean[column].astype(str).str.strip()

for column in text_columns["purchase_orders"]:
    purchase_orders_clean[column] = purchase_orders_clean[column].astype(str).str.strip()


# ------------------------------------------
# 3. Calculate actual supplier lead time
# ------------------------------------------

purchase_orders_clean["Actual_Lead_Time"] = (
    purchase_orders_clean["Arrival_Date"]
    - purchase_orders_clean["Order_Date"]
).dt.days


# ------------------------------------------
# 4. Create basic sales time features
# ------------------------------------------

sales_clean["Year"] = sales_clean["Sale_Date"].dt.year
sales_clean["Month"] = sales_clean["Sale_Date"].dt.month
sales_clean["Month_Name"] = sales_clean["Sale_Date"].dt.strftime("%b")
sales_clean["Quarter"] = sales_clean["Sale_Date"].dt.quarter


# ------------------------------------------
# 5. Validation
# ------------------------------------------

print("=" * 60)
print("ETL CLEANING VALIDATION")
print("=" * 60)

print("\nSales Clean:", sales_clean.shape)
print("Inventory Clean:", inventory_clean.shape)
print("Suppliers Clean:", suppliers_clean.shape)
print("Purchase Orders Clean:", purchase_orders_clean.shape)

print("\nDate conversion check:")
print("Sales invalid dates:",
      sales_clean["Sale_Date"].isna().sum())

print("Inventory invalid dates:",
      inventory_clean["Last_Updated"].isna().sum())

print("Purchase Order invalid dates:",
      purchase_orders_clean["Order_Date"].isna().sum())

print("Arrival invalid dates:",
      purchase_orders_clean["Arrival_Date"].isna().sum())

print("\nActual Lead Time:")
print(
    purchase_orders_clean["Actual_Lead_Time"]
    .value_counts()
    .sort_index()
)

print("\nSales preview:")
display(sales_clean.head())

print("\nPurchase Orders preview:")
display(purchase_orders_clean.head())

ETL CLEANING VALIDATION

Sales Clean: (5000, 10)
Inventory Clean: (200, 6)
Suppliers Clean: (50, 5)
Purchase Orders Clean: (1000, 7)

Date conversion check:
Sales invalid dates: 0
Inventory invalid dates: 0
Purchase Order invalid dates: 0
Arrival invalid dates: 0

Actual Lead Time:
Actual_Lead_Time
5    1000
Name: count, dtype: int64

Sales preview:


,Sale_ID,Product_ID,Store_ID,Sale_Date,Quantity_Sold,Revenue,Year,Month,Month_Name,Quarter
0,0d12b607,P001,S102,2024-01-01,322,2556,2024,1,Jan,1
1,2e13fe47,P003,S101,2024-01-02,28,42606,2024,1,Jan,1
2,c05afd2d,P003,S104,2024-01-03,292,19551,2024,1,Jan,1
3,aff62361,P004,S101,2024-01-04,280,26625,2024,1,Jan,1
4,9d190af1,P003,S101,2024-01-05,353,24001,2024,1,Jan,1



Purchase Orders preview:


,Order_ID,Product_ID,Supplier_ID,Order_Date,Quantity,Arrival_Date,Actual_Lead_Time
0,PO0001,P001,SUP002,2024-01-01,158,2024-01-06,5
1,PO0002,P005,SUP019,2024-01-02,440,2024-01-07,5
2,PO0003,P002,SUP036,2024-01-03,423,2024-01-08,5
3,PO0004,P004,SUP021,2024-01-04,88,2024-01-09,5
4,PO0005,P001,SUP002,2024-01-05,329,2024-01-10,5


In [10]:
# ==========================================
# BLOCK 8: ETL RELATIONSHIP VALIDATION
# ==========================================

print("=" * 60)
print("RELATIONSHIP VALIDATION")
print("=" * 60)

# ------------------------------------------
# Product relationships
# ------------------------------------------

sales_products = set(sales_clean["Product_ID"])
inventory_products = set(inventory_clean["Product_ID"])
supplier_products = set(suppliers_clean["Product_ID"])
purchase_products = set(purchase_orders_clean["Product_ID"])

print("\nPRODUCT COVERAGE")
print("-" * 40)

print("Products in Sales:", len(sales_products))
print("Products in Inventory:", len(inventory_products))
print("Products in Suppliers:", len(supplier_products))
print("Products in Purchase Orders:", len(purchase_products))

print("\nProducts in Sales but not Inventory:",
      sorted(sales_products - inventory_products))

print("Products in Sales but not Suppliers:",
      sorted(sales_products - supplier_products))

print("Products in Sales but not Purchase Orders:",
      sorted(sales_products - purchase_products))


# ------------------------------------------
# Supplier relationships
# ------------------------------------------

supplier_ids = set(suppliers_clean["Supplier_ID"])
po_supplier_ids = set(purchase_orders_clean["Supplier_ID"])

print("\nSUPPLIER COVERAGE")
print("-" * 40)

print("Suppliers in Supplier Master:", len(supplier_ids))
print("Suppliers in Purchase Orders:", len(po_supplier_ids))

print("\nSuppliers in Purchase Orders but not Supplier Master:",
      sorted(po_supplier_ids - supplier_ids))

print("Suppliers in Supplier Master but not Purchase Orders:",
      sorted(supplier_ids - po_supplier_ids))


# ------------------------------------------
# Store relationships
# ------------------------------------------

sales_stores = set(sales_clean["Store_ID"])
inventory_stores = set(inventory_clean["Store_ID"])

print("\nSTORE COVERAGE")
print("-" * 40)

print("Stores in Sales:", len(sales_stores))
print("Stores in Inventory:", len(inventory_stores))

print("\nStores in Sales but not Inventory:",
      sorted(sales_stores - inventory_stores))

print("Stores in Inventory but not Sales:",
      sorted(inventory_stores - sales_stores))


# ------------------------------------------
# Supplier-product combinations
# ------------------------------------------

supplier_product_pairs = (
    suppliers_clean[["Supplier_ID", "Product_ID"]]
    .drop_duplicates()
)

po_supplier_product_pairs = (
    purchase_orders_clean[["Supplier_ID", "Product_ID"]]
    .drop_duplicates()
)

print("\nSUPPLIER-PRODUCT RELATIONSHIP")
print("-" * 40)

print("Unique Supplier-Product pairs in Supplier Master:",
      len(supplier_product_pairs))

print("Unique Supplier-Product pairs in Purchase Orders:",
      len(po_supplier_product_pairs))

print("\nPurchase-order pairs missing from Supplier Master:")
missing_pairs = (
    po_supplier_product_pairs
    .merge(
        supplier_product_pairs,
        on=["Supplier_ID", "Product_ID"],
        how="left",
        indicator=True
    )
)

display(
    missing_pairs[
        missing_pairs["_merge"] == "left_only"
    ].drop(columns="_merge")
)

RELATIONSHIP VALIDATION

PRODUCT COVERAGE
----------------------------------------
Products in Sales: 5
Products in Inventory: 5
Products in Suppliers: 5
Products in Purchase Orders: 5

Products in Sales but not Inventory: []
Products in Sales but not Suppliers: []
Products in Sales but not Purchase Orders: []

SUPPLIER COVERAGE
----------------------------------------
Suppliers in Supplier Master: 50
Suppliers in Purchase Orders: 50

Suppliers in Purchase Orders but not Supplier Master: []
Suppliers in Supplier Master but not Purchase Orders: []

STORE COVERAGE
----------------------------------------
Stores in Sales: 4
Stores in Inventory: 4

Stores in Sales but not Inventory: []
Stores in Inventory but not Sales: []

SUPPLIER-PRODUCT RELATIONSHIP
----------------------------------------
Unique Supplier-Product pairs in Supplier Master: 50
Unique Supplier-Product pairs in Purchase Orders: 246

Purchase-order pairs missing from Supplier Master:


,Supplier_ID,Product_ID
0,SUP002,P001
1,SUP019,P005
2,SUP036,P002
3,SUP021,P004
4,SUP040,P001
...,...,...
239,SUP015,P003
240,SUP043,P001
241,SUP023,P004
242,SUP037,P005


In [11]:
# ==========================================
# BLOCK 8A: STORE & SUPPLIER RELATIONSHIP SUMMARY
# ==========================================

print("=" * 60)
print("STORE COVERAGE")
print("=" * 60)

sales_stores = set(sales_clean["Store_ID"])
inventory_stores = set(inventory_clean["Store_ID"])

print("Stores in Sales:", len(sales_stores))
print("Stores in Inventory:", len(inventory_stores))

print("\nStores in Sales but not Inventory:")
print(sorted(sales_stores - inventory_stores))

print("\nStores in Inventory but not Sales:")
print(sorted(inventory_stores - sales_stores))


print("\n" + "=" * 60)
print("SUPPLIER-PRODUCT RELATIONSHIP SUMMARY")
print("=" * 60)

supplier_pairs = (
    suppliers_clean[["Supplier_ID", "Product_ID"]]
    .drop_duplicates()
)

po_pairs = (
    purchase_orders_clean[["Supplier_ID", "Product_ID"]]
    .drop_duplicates()
)

missing_pairs = (
    po_pairs
    .merge(
        supplier_pairs,
        on=["Supplier_ID", "Product_ID"],
        how="left",
        indicator=True
    )
)

print("Supplier master pairs:", len(supplier_pairs))
print("Purchase-order pairs:", len(po_pairs))
print(
    "PO pairs also present in supplier master:",
    (missing_pairs["_merge"] == "both").sum()
)
print(
    "PO pairs absent from supplier master:",
    (missing_pairs["_merge"] == "left_only").sum()
)

STORE COVERAGE
Stores in Sales: 4
Stores in Inventory: 4

Stores in Sales but not Inventory:
[]

Stores in Inventory but not Sales:
[]

SUPPLIER-PRODUCT RELATIONSHIP SUMMARY
Supplier master pairs: 50
Purchase-order pairs: 246
PO pairs also present in supplier master: 48
PO pairs absent from supplier master: 198


In [12]:
# ==========================================
# BLOCK 9: DAILY DEMAND AGGREGATION
# ==========================================

daily_demand = (
    sales_clean
    .groupby(
        ["Sale_Date", "Product_ID", "Store_ID"],
        as_index=False
    )
    .agg(
        Quantity_Sold=("Quantity_Sold", "sum"),
        Revenue=("Revenue", "sum"),
        Transaction_Count=("Sale_ID", "nunique")
    )
    .sort_values(
        ["Sale_Date", "Product_ID", "Store_ID"]
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("DAILY DEMAND DATASET")
print("=" * 60)

print("\nShape:", daily_demand.shape)

print("\nColumns:")
print(daily_demand.columns.tolist())

print("\nDate range:")
print("First:", daily_demand["Sale_Date"].min())
print("Last :", daily_demand["Sale_Date"].max())

print("\nUnique Products:",
      daily_demand["Product_ID"].nunique())

print("Unique Stores:",
      daily_demand["Store_ID"].nunique())

print("\nTotal Quantity Sold:",
      daily_demand["Quantity_Sold"].sum())

print("Total Revenue:",
      daily_demand["Revenue"].sum())

print("\nPreview:")
display(daily_demand.head(10))

DAILY DEMAND DATASET

Shape: (5000, 6)

Columns:
['Sale_Date', 'Product_ID', 'Store_ID', 'Quantity_Sold', 'Revenue', 'Transaction_Count']

Date range:
First: 2024-01-01 00:00:00
Last : 2037-09-08 00:00:00

Unique Products: 5
Unique Stores: 4

Total Quantity Sold: 1292122
Total Revenue: 125870014

Preview:


,Sale_Date,Product_ID,Store_ID,Quantity_Sold,Revenue,Transaction_Count
0,2024-01-01,P001,S102,322,2556,1
1,2024-01-02,P003,S101,28,42606,1
2,2024-01-03,P003,S104,292,19551,1
3,2024-01-04,P004,S101,280,26625,1
4,2024-01-05,P003,S101,353,24001,1
5,2024-01-06,P004,S101,227,7400,1
6,2024-01-07,P003,S102,101,28456,1
7,2024-01-08,P005,S104,499,1218,1
8,2024-01-09,P004,S103,34,28115,1
9,2024-01-10,P002,S103,25,6962,1


In [13]:
# ==========================================
# BLOCK 10: MONTHLY DEMAND AGGREGATION
# ==========================================

monthly_demand = (
    sales_clean
    .assign(
        Month_Date=sales_clean["Sale_Date"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby(
        ["Month_Date", "Product_ID"],
        as_index=False
    )
    .agg(
        Quantity_Sold=("Quantity_Sold", "sum"),
        Revenue=("Revenue", "sum"),
        Transaction_Count=("Sale_ID", "nunique"),
        Store_Count=("Store_ID", "nunique")
    )
    .sort_values(
        ["Month_Date", "Product_ID"]
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("MONTHLY DEMAND DATASET")
print("=" * 60)

print("\nShape:", monthly_demand.shape)

print("\nDate range:")
print("First:", monthly_demand["Month_Date"].min())
print("Last :", monthly_demand["Month_Date"].max())

print("\nUnique Products:",
      monthly_demand["Product_ID"].nunique())

print("\nNumber of Months:",
      monthly_demand["Month_Date"].nunique())

print("\nTotal Quantity Sold:",
      monthly_demand["Quantity_Sold"].sum())

print("Total Revenue:",
      monthly_demand["Revenue"].sum())

print("\nPreview:")
display(monthly_demand.head(10))

MONTHLY DEMAND DATASET

Shape: (823, 6)

Date range:
First: 2024-01-01 00:00:00
Last : 2037-09-01 00:00:00

Unique Products: 5

Number of Months: 165

Total Quantity Sold: 1292122
Total Revenue: 125870014

Preview:


,Month_Date,Product_ID,Quantity_Sold,Revenue,Transaction_Count,Store_Count
0,2024-01-01,P001,902,97996,4,4
1,2024-01-01,P002,361,60578,3,3
2,2024-01-01,P003,2518,199049,10,4
3,2024-01-01,P004,2257,195525,8,4
4,2024-01-01,P005,2585,153168,6,4
5,2024-02-01,P001,1322,158814,5,4
6,2024-02-01,P002,2678,234072,9,4
7,2024-02-01,P003,817,121200,5,2
8,2024-02-01,P004,1342,87532,4,3
9,2024-02-01,P005,1175,147292,6,4


In [14]:
# ==========================================
# BLOCK 11: PRODUCT-MONTH COMPLETENESS
# ==========================================

print("=" * 60)
print("PRODUCT-MONTH COMPLETENESS CHECK")
print("=" * 60)

# Expected complete product-month combinations
expected_combinations = (
    pd.MultiIndex.from_product(
        [
            monthly_demand["Month_Date"].unique(),
            monthly_demand["Product_ID"].unique()
        ],
        names=["Month_Date", "Product_ID"]
    )
    .to_frame(index=False)
)

# Find missing combinations
missing_product_months = (
    expected_combinations
    .merge(
        monthly_demand[["Month_Date", "Product_ID"]],
        on=["Month_Date", "Product_ID"],
        how="left",
        indicator=True
    )
)

missing_product_months = missing_product_months[
    missing_product_months["_merge"] == "left_only"
].drop(columns="_merge")

print("\nExpected product-month combinations:",
      len(expected_combinations))

print("Actual product-month combinations:",
      len(monthly_demand))

print("Missing product-month combinations:",
      len(missing_product_months))

print("\nMissing combinations:")
display(missing_product_months)

print("\nRecords per product:")
print(
    monthly_demand
    .groupby("Product_ID")["Month_Date"]
    .nunique()
    .sort_index()
)

PRODUCT-MONTH COMPLETENESS CHECK

Expected product-month combinations: 825
Actual product-month combinations: 823
Missing product-month combinations: 2

Missing combinations:


,Month_Date,Product_ID
170,2026-11-01,P001
821,2037-09-01,P002



Records per product:
Product_ID
P001    164
P002    164
P003    165
P004    165
P005    165
Name: Month_Date, dtype: int64


In [15]:
# ==========================================
# BLOCK 12: PRODUCT DEMAND PROFILE
# ==========================================

product_demand_profile = (
    sales_clean
    .groupby("Product_ID", as_index=False)
    .agg(
        Total_Quantity_Sold=("Quantity_Sold", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Sales_Records=("Sale_ID", "nunique"),
        Average_Quantity_Per_Sale=("Quantity_Sold", "mean"),
        Average_Revenue_Per_Sale=("Revenue", "mean")
    )
)

product_demand_profile["Average_Quantity_Per_Sale"] = (
    product_demand_profile["Average_Quantity_Per_Sale"].round(2)
)

product_demand_profile["Average_Revenue_Per_Sale"] = (
    product_demand_profile["Average_Revenue_Per_Sale"].round(2)
)

product_demand_profile = product_demand_profile.sort_values(
    "Total_Quantity_Sold",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("PRODUCT DEMAND PROFILE")
print("=" * 60)

display(product_demand_profile)

print("\nFast-moving product based on total quantity:")
print(product_demand_profile.iloc[0]["Product_ID"])

print("\nSlow-moving product based on total quantity:")
print(product_demand_profile.iloc[-1]["Product_ID"])

PRODUCT DEMAND PROFILE


,Product_ID,Total_Quantity_Sold,Total_Revenue,Sales_Records,Average_Quantity_Per_Sale,Average_Revenue_Per_Sale
0,P002,269100,26084882,1018,264.34,25623.66
1,P001,260296,25355109,1044,249.33,24286.50
2,P004,258635,24835649,985,262.57,25213.86
3,P005,254794,24896449,973,261.86,25587.31
4,P003,249297,24697925,980,254.38,25201.96



Fast-moving product based on total quantity:
P002

Slow-moving product based on total quantity:
P003


In [16]:
# ==========================================
# BLOCK 13: INVENTORY STATUS ANALYSIS
# ==========================================

inventory_analysis = inventory_clean.copy()

inventory_analysis["Stock_Status"] = np.where(
    inventory_analysis["Stock_Level"]
    < inventory_analysis["Reorder_Level"],
    "Below Reorder Level",
    "At/Above Reorder Level"
)

inventory_analysis["Stock_Gap"] = (
    inventory_analysis["Stock_Level"]
    - inventory_analysis["Reorder_Level"]
)

print("=" * 60)
print("INVENTORY STATUS ANALYSIS")
print("=" * 60)

print("\nTotal inventory snapshots:",
      len(inventory_analysis))

print("\nStock status:")
print(
    inventory_analysis["Stock_Status"]
    .value_counts()
)

print("\nSnapshots below reorder level:",
      (inventory_analysis["Stock_Status"] == "Below Reorder Level").sum())

print("\nSnapshots at/above reorder level:",
      (inventory_analysis["Stock_Status"] == "At/Above Reorder Level").sum())

print("\nAverage stock level:",
      round(inventory_analysis["Stock_Level"].mean(), 2))

print("Average reorder level:",
      round(inventory_analysis["Reorder_Level"].mean(), 2))

print("\nInventory analysis preview:")
display(
    inventory_analysis[
        [
            "Product_ID",
            "Store_ID",
            "Warehouse_ID",
            "Stock_Level",
            "Reorder_Level",
            "Stock_Status",
            "Stock_Gap",
            "Last_Updated"
        ]
    ].head(10)
)

INVENTORY STATUS ANALYSIS

Total inventory snapshots: 200

Stock status:
Stock_Status
At/Above Reorder Level    147
Below Reorder Level        53
Name: count, dtype: int64

Snapshots below reorder level: 53

Snapshots at/above reorder level: 147

Average stock level: 224.96
Average reorder level: 124.79

Inventory analysis preview:


,Product_ID,Store_ID,Warehouse_ID,Stock_Level,Reorder_Level,Stock_Status,Stock_Gap,Last_Updated
0,P005,S101,W003,498,163,At/Above Reorder Level,335,2024-02-01
1,P004,S103,W003,24,178,Below Reorder Level,-154,2024-02-02
2,P002,S101,W002,245,188,At/Above Reorder Level,57,2024-02-03
3,P003,S101,W003,61,148,Below Reorder Level,-87,2024-02-04
4,P002,S101,W002,57,155,Below Reorder Level,-98,2024-02-05
5,P003,S103,W002,443,63,At/Above Reorder Level,380,2024-02-06
6,P005,S102,W002,392,105,At/Above Reorder Level,287,2024-02-07
7,P004,S101,W002,284,124,At/Above Reorder Level,160,2024-02-08
8,P004,S102,W001,342,169,At/Above Reorder Level,173,2024-02-09
9,P004,S102,W003,17,67,Below Reorder Level,-50,2024-02-10


In [17]:
# ==========================================
# BLOCK 14: PRODUCT DAILY DEMAND RATE
# ==========================================

product_daily_demand = (
    sales_clean
    .groupby("Product_ID", as_index=False)
    .agg(
        Total_Quantity_Sold=("Quantity_Sold", "sum"),
        First_Sale_Date=("Sale_Date", "min"),
        Last_Sale_Date=("Sale_Date", "max")
    )
)

product_daily_demand["Days_Covered"] = (
    product_daily_demand["Last_Sale_Date"]
    - product_daily_demand["First_Sale_Date"]
).dt.days + 1

product_daily_demand["Average_Daily_Demand"] = (
    product_daily_demand["Total_Quantity_Sold"]
    / product_daily_demand["Days_Covered"]
)

product_daily_demand["Average_Daily_Demand"] = (
    product_daily_demand["Average_Daily_Demand"].round(2)
)

print("=" * 60)
print("PRODUCT DAILY DEMAND RATE")
print("=" * 60)

display(product_daily_demand)

PRODUCT DAILY DEMAND RATE


,Product_ID,Total_Quantity_Sold,First_Sale_Date,Last_Sale_Date,Days_Covered,Average_Daily_Demand
0,P001,260296,2024-01-01,2037-09-08,5000,52.06
1,P002,269100,2024-01-10,2037-08-25,4977,54.07
2,P003,249297,2024-01-02,2037-09-05,4996,49.90
3,P004,258635,2024-01-04,2037-09-07,4996,51.77
4,P005,254794,2024-01-08,2037-09-02,4987,51.09


In [18]:
# ==========================================
# BLOCK 15: SUPPLIER PERFORMANCE PROFILE
# ==========================================

supplier_order_summary = (
    purchase_orders_clean
    .groupby("Supplier_ID", as_index=False)
    .agg(
        Total_Orders=("Order_ID", "nunique"),
        Total_Quantity_Ordered=("Quantity", "sum"),
        Average_Order_Quantity=("Quantity", "mean"),
        Average_Actual_Lead_Time=("Actual_Lead_Time", "mean"),
        Products_Supplied=("Product_ID", "nunique")
    )
)

supplier_performance = suppliers_clean[
    [
        "Supplier_ID",
        "Supplier_Name",
        "Lead_Time (days)",
        "Order_Frequency"
    ]
].merge(
    supplier_order_summary,
    on="Supplier_ID",
    how="left"
)

supplier_performance["Average_Order_Quantity"] = (
    supplier_performance["Average_Order_Quantity"].round(2)
)

supplier_performance["Average_Actual_Lead_Time"] = (
    supplier_performance["Average_Actual_Lead_Time"].round(2)
)

supplier_performance = supplier_performance.sort_values(
    "Lead_Time (days)"
).reset_index(drop=True)

print("=" * 60)
print("SUPPLIER PERFORMANCE PROFILE")
print("=" * 60)

print("\nShape:", supplier_performance.shape)

print("\nSuppliers without purchase orders:",
      supplier_performance["Total_Orders"].isna().sum())

print("\nAverage supplier-reported lead time:",
      round(supplier_performance["Lead_Time (days)"].mean(), 2))

print("\nAverage actual PO lead time:",
      round(supplier_performance["Average_Actual_Lead_Time"].mean(), 2))

print("\nPreview:")
display(supplier_performance.head(10))

SUPPLIER PERFORMANCE PROFILE

Shape: (50, 9)

Suppliers without purchase orders: 0

Average supplier-reported lead time: 6.22

Average actual PO lead time: 5.0

Preview:


,Supplier_ID,Supplier_Name,Lead_Time (days),Order_Frequency,Total_Orders,Total_Quantity_Ordered,Average_Order_Quantity,Average_Actual_Lead_Time,Products_Supplied
0,SUP028,Taylor Ltd,3,Biweekly,23,6299,273.87,5.0,5
1,SUP032,Ferguson Group,3,Weekly,21,4668,222.29,5.0,5
2,SUP041,Brown Ltd,3,Weekly,14,4114,293.86,5.0,5
3,SUP039,Ortega-Manning,3,Monthly,22,6557,298.05,5.0,5
4,SUP024,Thompson-Hebert,4,Biweekly,19,5075,267.11,5.0,5
5,SUP019,Bowers and Sons,4,Weekly,20,5098,254.90,5.0,5
6,SUP003,Acosta Inc,4,Monthly,22,5521,250.95,5.0,5
7,SUP015,Watson Ltd,4,Biweekly,16,4556,284.75,5.0,5
8,SUP021,Robertson LLC,4,Biweekly,19,5274,277.58,5.0,5
9,SUP022,"Alvarado, Ramirez and Taylor",4,Biweekly,17,4523,266.06,5.0,5


In [19]:
# ==========================================
# BLOCK 16: SUPPLIER CLUSTERING FEATURES
# ==========================================

supplier_clustering = supplier_performance[
    [
        "Supplier_ID",
        "Supplier_Name",
        "Lead_Time (days)",
        "Order_Frequency"
    ]
].copy()

# Convert order frequency into numeric frequency score
frequency_mapping = {
    "Weekly": 3,
    "Biweekly": 2,
    "Monthly": 1
}

supplier_clustering["Order_Frequency_Score"] = (
    supplier_clustering["Order_Frequency"]
    .map(frequency_mapping)
)

print("=" * 60)
print("SUPPLIER CLUSTERING FEATURES")
print("=" * 60)

print("\nFrequency mapping:")
print(frequency_mapping)

print("\nMissing frequency scores:",
      supplier_clustering["Order_Frequency_Score"].isna().sum())

print("\nFeature ranges:")
print(
    supplier_clustering[
        [
            "Lead_Time (days)",
            "Order_Frequency_Score"
        ]
    ].agg(["min", "max", "mean"]).round(2)
)

print("\nClustering dataset:")
display(supplier_clustering.head(10))

SUPPLIER CLUSTERING FEATURES

Frequency mapping:
{'Weekly': 3, 'Biweekly': 2, 'Monthly': 1}

Missing frequency scores: 0

Feature ranges:
      Lead_Time (days)  Order_Frequency_Score
min               3.00                    1.0
max               9.00                    3.0
mean              6.22                    2.1

Clustering dataset:


,Supplier_ID,Supplier_Name,Lead_Time (days),Order_Frequency,Order_Frequency_Score
0,SUP028,Taylor Ltd,3,Biweekly,2
1,SUP032,Ferguson Group,3,Weekly,3
2,SUP041,Brown Ltd,3,Weekly,3
3,SUP039,Ortega-Manning,3,Monthly,1
4,SUP024,Thompson-Hebert,4,Biweekly,2
5,SUP019,Bowers and Sons,4,Weekly,3
6,SUP003,Acosta Inc,4,Monthly,1
7,SUP015,Watson Ltd,4,Biweekly,2
8,SUP021,Robertson LLC,4,Biweekly,2
9,SUP022,"Alvarado, Ramirez and Taylor",4,Biweekly,2


In [20]:
# ==========================================
# BLOCK 17: K-MEANS SUPPLIER CLUSTERING
# ==========================================

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Select clustering features
cluster_features = supplier_clustering[
    [
        "Lead_Time (days)",
        "Order_Frequency_Score"
    ]
].copy()

# Standardize features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(cluster_features)

# K-Means clustering
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

supplier_clustering["Cluster"] = kmeans.fit_predict(X_scaled)

# Cluster centers in original scale
cluster_centers_scaled = kmeans.cluster_centers_

cluster_centers = scaler.inverse_transform(
    cluster_centers_scaled
)

cluster_summary = pd.DataFrame(
    cluster_centers,
    columns=[
        "Lead_Time (days)",
        "Order_Frequency_Score"
    ]
)

cluster_summary["Cluster"] = cluster_summary.index

cluster_summary = cluster_summary[
    [
        "Cluster",
        "Lead_Time (days)",
        "Order_Frequency_Score"
    ]
]

cluster_summary["Lead_Time (days)"] = (
    cluster_summary["Lead_Time (days)"].round(2)
)

cluster_summary["Order_Frequency_Score"] = (
    cluster_summary["Order_Frequency_Score"].round(2)
)

print("=" * 60)
print("K-MEANS SUPPLIER CLUSTERING")
print("=" * 60)

print("\nCluster distribution:")
print(
    supplier_clustering["Cluster"]
    .value_counts()
    .sort_index()
)

print("\nCluster centers:")
display(cluster_summary)

print("\nSupplier clustering preview:")
display(
    supplier_clustering[
        [
            "Supplier_ID",
            "Supplier_Name",
            "Lead_Time (days)",
            "Order_Frequency",
            "Order_Frequency_Score",
            "Cluster"
        ]
    ].head(15)
)

K-MEANS SUPPLIER CLUSTERING

Cluster distribution:
Cluster
0    18
1    18
2    14
Name: count, dtype: int64

Cluster centers:


,Cluster,Lead_Time (days),Order_Frequency_Score
0,0,4.28,2.44
1,1,8.33,2.61
2,2,6.00,1.00



Supplier clustering preview:


,Supplier_ID,Supplier_Name,Lead_Time (days),Order_Frequency,Order_Frequency_Score,Cluster
0,SUP028,Taylor Ltd,3,Biweekly,2,0
1,SUP032,Ferguson Group,3,Weekly,3,0
2,SUP041,Brown Ltd,3,Weekly,3,0
3,SUP039,Ortega-Manning,3,Monthly,1,2
4,SUP024,Thompson-Hebert,4,Biweekly,2,0
5,SUP019,Bowers and Sons,4,Weekly,3,0
6,SUP003,Acosta Inc,4,Monthly,1,2
7,SUP015,Watson Ltd,4,Biweekly,2,0
8,SUP021,Robertson LLC,4,Biweekly,2,0
9,SUP022,"Alvarado, Ramirez and Taylor",4,Biweekly,2,0


In [21]:
# ==========================================
# BLOCK 18: SUPPLIER CLUSTER PROFILES
# ==========================================

# Add cluster-level supplier counts
cluster_counts = (
    supplier_clustering["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Supplier_Count")
    .reset_index()
    .rename(columns={"index": "Cluster"})
)

# Merge counts with cluster centers
cluster_profiles = cluster_summary.merge(
    cluster_counts,
    on="Cluster",
    how="left"
)

# Create factual descriptive profiles
def describe_cluster(row):
    lead_time = row["Lead_Time (days)"]
    frequency = row["Order_Frequency_Score"]

    if lead_time <= 5 and frequency >= 2:
        return "Shorter lead time / more frequent ordering"
    elif lead_time > 7:
        return "Longer lead time / more frequent ordering"
    else:
        return "Medium lead time / monthly ordering"

cluster_profiles["Cluster_Profile"] = (
    cluster_profiles.apply(describe_cluster, axis=1)
)

print("=" * 60)
print("SUPPLIER CLUSTER PROFILES")
print("=" * 60)

display(
    cluster_profiles[
        [
            "Cluster",
            "Supplier_Count",
            "Lead_Time (days)",
            "Order_Frequency_Score",
            "Cluster_Profile"
        ]
    ]
)

SUPPLIER CLUSTER PROFILES


,Cluster,Supplier_Count,Lead_Time (days),Order_Frequency_Score,Cluster_Profile
0,0,18,4.28,2.44,Shorter lead time / more frequent ordering
1,1,18,8.33,2.61,Longer lead time / more frequent ordering
2,2,14,6.00,1.00,Medium lead time / monthly ordering


In [22]:
# ==========================================
# BLOCK 19: MONTHLY DEMAND TREND ANALYSIS
# ==========================================

monthly_total_demand = (
    monthly_demand
    .groupby("Month_Date", as_index=False)
    .agg(
        Total_Quantity_Sold=("Quantity_Sold", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Product_Count=("Product_ID", "nunique")
    )
    .sort_values("Month_Date")
    .reset_index(drop=True)
)

print("=" * 60)
print("MONTHLY DEMAND TREND")
print("=" * 60)

print("\nShape:", monthly_total_demand.shape)

print("\nFirst 10 months:")
display(monthly_total_demand.head(10))

print("\nLast 10 months:")
display(monthly_total_demand.tail(10))

print("\nMinimum monthly quantity:",
      monthly_total_demand["Total_Quantity_Sold"].min())

print("Maximum monthly quantity:",
      monthly_total_demand["Total_Quantity_Sold"].max())

print("Average monthly quantity:",
      round(
          monthly_total_demand["Total_Quantity_Sold"].mean(),
          2
      )
      )

print("\nMonths with fewer than 5 products:")
display(
    monthly_total_demand[
        monthly_total_demand["Product_Count"] < 5
    ]
)

MONTHLY DEMAND TREND

Shape: (165, 4)

First 10 months:


,Month_Date,Total_Quantity_Sold,Total_Revenue,Product_Count
0,2024-01-01,8623,706316,5
1,2024-02-01,7334,748910,5
2,2024-03-01,7327,749353,5
3,2024-04-01,8389,770649,5
4,2024-05-01,8277,793790,5
5,2024-06-01,9040,775810,5
6,2024-07-01,7652,801268,5
7,2024-08-01,8210,780492,5
8,2024-09-01,7401,823266,5
9,2024-10-01,8151,594306,5



Last 10 months:


,Month_Date,Total_Quantity_Sold,Total_Revenue,Product_Count
155,2036-12-01,9108,799441,5
156,2037-01-01,7927,812684,5
157,2037-02-01,8434,635952,5
158,2037-03-01,7043,701480,5
159,2037-04-01,7179,720944,5
160,2037-05-01,8826,734901,5
161,2037-06-01,7585,664893,5
162,2037-07-01,6738,696360,5
163,2037-08-01,8385,850017,5
164,2037-09-01,1782,287878,4



Minimum monthly quantity: 1782
Maximum monthly quantity: 9509
Average monthly quantity: 7831.04

Months with fewer than 5 products:


,Month_Date,Total_Quantity_Sold,Total_Revenue,Product_Count
34,2026-11-01,7135,743571,4
164,2037-09-01,1782,287878,4


In [23]:
# ==========================================
# BLOCK 20: FORECASTING DATA PREPARATION
# ==========================================

# Identify months containing all products
complete_months = (
    monthly_total_demand[
        monthly_total_demand["Product_Count"] == 5
    ]["Month_Date"]
    .sort_values()
    .tolist()
)

last_complete_month = complete_months[-1]

print("=" * 60)
print("FORECASTING DATA PREPARATION")
print("=" * 60)

print("\nLast complete month:", last_complete_month)

print("\nIncomplete months excluded from forecasting:")
display(
    monthly_total_demand[
        monthly_total_demand["Product_Count"] < 5
    ]
)

# Keep monthly product data only through the last complete month
forecast_history = monthly_demand[
    monthly_demand["Month_Date"] <= last_complete_month
].copy()

print("\nForecast history shape:", forecast_history.shape)

print("Forecast history start:",
      forecast_history["Month_Date"].min())

print("Forecast history end:",
      forecast_history["Month_Date"].max())

print("\nProducts in last complete month:")

print(
    forecast_history[
        forecast_history["Month_Date"] == last_complete_month
    ]["Product_ID"].tolist()
)

print("\nLast complete month demand:")
display(
    forecast_history[
        forecast_history["Month_Date"] == last_complete_month
    ].sort_values("Product_ID")
)

FORECASTING DATA PREPARATION

Last complete month: 2037-08-01 00:00:00

Incomplete months excluded from forecasting:


,Month_Date,Total_Quantity_Sold,Total_Revenue,Product_Count
34,2026-11-01,7135,743571,4
164,2037-09-01,1782,287878,4



Forecast history shape: (819, 6)
Forecast history start: 2024-01-01 00:00:00
Forecast history end: 2037-08-01 00:00:00

Products in last complete month:
['P001', 'P002', 'P003', 'P004', 'P005']

Last complete month demand:


,Month_Date,Product_ID,Quantity_Sold,Revenue,Transaction_Count,Store_Count
814,2037-08-01,P001,1245,98905,5,2
815,2037-08-01,P002,1707,222328,7,4
816,2037-08-01,P003,1519,207718,6,3
817,2037-08-01,P004,2830,221930,9,4
818,2037-08-01,P005,1084,99136,4,3


In [24]:
# ==========================================
# BLOCK 21: BASELINE NEXT-MONTH FORECAST
# ==========================================

# Get the last 3 complete months
last_3_months = (
    forecast_history["Month_Date"]
    .drop_duplicates()
    .sort_values()
    .tail(3)
    .tolist()
)

print("=" * 60)
print("BASELINE FORECAST")
print("=" * 60)

print("\nLast 3 complete months:")
for month in last_3_months:
    print(month.strftime("%Y-%m"))

# Filter to last 3 months
recent_demand = forecast_history[
    forecast_history["Month_Date"].isin(last_3_months)
].copy()

# Calculate baseline forecast
baseline_forecast = (
    recent_demand
    .groupby("Product_ID", as_index=False)
    .agg(
        Baseline_Forecast=("Quantity_Sold", "mean")
    )
)

baseline_forecast["Baseline_Forecast"] = (
    baseline_forecast["Baseline_Forecast"]
    .round(2)
)

print("\nBaseline forecast:")
display(baseline_forecast)

print("\nTotal baseline forecast:",
      round(baseline_forecast["Baseline_Forecast"].sum(), 2))

BASELINE FORECAST

Last 3 complete months:
2037-06
2037-07
2037-08

Baseline forecast:


,Product_ID,Baseline_Forecast
0,P001,1169.67
1,P002,1484.67
2,P003,1597.33
3,P004,1851.67
4,P005,1466.00



Total baseline forecast: 7569.34


In [25]:
# ==========================================
# BLOCK 22: REGRESSION DEMAND FORECAST
# ==========================================

from sklearn.linear_model import LinearRegression

forecast_results = []

# Create sequential time index
forecast_months = (
    forecast_history["Month_Date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

month_to_index = {
    month: index
    for index, month in enumerate(forecast_months)
}

forecast_history_model = forecast_history.copy()

forecast_history_model["Time_Index"] = (
    forecast_history_model["Month_Date"]
    .map(month_to_index)
)

next_time_index = len(forecast_months)

# Train one regression model per product
for product in sorted(
    forecast_history_model["Product_ID"].unique()
):
    
    product_data = forecast_history_model[
        forecast_history_model["Product_ID"] == product
    ].sort_values("Time_Index")

    X = product_data[["Time_Index"]]
    y = product_data["Quantity_Sold"]

    model = LinearRegression()
    model.fit(X, y)

    prediction = model.predict(
        [[next_time_index]]
    )[0]

    # Demand cannot be negative
    prediction = max(0, prediction)

    forecast_results.append(
        {
            "Product_ID": product,
            "Regression_Forecast": round(prediction, 2),
            "Model_Slope": round(model.coef_[0], 4),
            "Model_Intercept": round(model.intercept_, 2)
        }
    )

regression_forecast = pd.DataFrame(
    forecast_results
)

# Compare against baseline
forecast_comparison = regression_forecast.merge(
    baseline_forecast,
    on="Product_ID",
    how="left"
)

forecast_comparison["Difference"] = (
    forecast_comparison["Regression_Forecast"]
    - forecast_comparison["Baseline_Forecast"]
).round(2)

forecast_comparison["Difference_Percent"] = (
    (
        forecast_comparison["Regression_Forecast"]
        - forecast_comparison["Baseline_Forecast"]
    )
    / forecast_comparison["Baseline_Forecast"]
    * 100
).round(2)

print("=" * 60)
print("REGRESSION DEMAND FORECAST")
print("=" * 60)

print("\nForecast month:",
      (
          last_complete_month
          + pd.DateOffset(months=1)
      ).strftime("%Y-%m"))

display(forecast_comparison)

print("\nTotal regression forecast:",
      round(
          forecast_comparison["Regression_Forecast"].sum(),
          2
      ))

print("\nTotal baseline forecast:",
      round(
          forecast_comparison["Baseline_Forecast"].sum(),
          2
      ))

REGRESSION DEMAND FORECAST

Forecast month: 2037-09


c:\Users\MKDco\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\MKDco\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\MKDco\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\MKDco\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\MKDco\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py

,Product_ID,Regression_Forecast,Model_Slope,Model_Intercept,Baseline_Forecast,Difference,Difference_Percent
0,P001,1511.64,-1.0094,1677.19,1169.67,341.97,29.24
1,P002,1684.44,0.5283,1597.80,1484.67,199.77,13.46
2,P003,1474.91,-0.5186,1559.97,1597.33,-122.42,-7.66
3,P004,1606.75,0.4152,1538.66,1851.67,-244.92,-13.23
4,P005,1622.70,0.8573,1482.10,1466.00,156.70,10.69



Total regression forecast: 7900.44

Total baseline forecast: 7569.34


In [26]:
# ==========================================
# BLOCK 23: TIME-SERIES MODEL VALIDATION
# ==========================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use the final 3 complete months as validation period
all_months = (
    forecast_history["Month_Date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

validation_months = all_months.tail(3).tolist()
training_months = all_months[
    ~all_months.isin(validation_months)
]

validation_predictions = []

for product in sorted(
    forecast_history["Product_ID"].unique()
):

    product_data = forecast_history[
        forecast_history["Product_ID"] == product
    ].copy()

    product_data = product_data.sort_values("Month_Date")

    # Create time index based on complete monthly sequence
    product_data["Time_Index"] = (
        product_data["Month_Date"]
        .map(month_to_index)
    )

    train_data = product_data[
        product_data["Month_Date"].isin(training_months)
    ]

    test_data = product_data[
        product_data["Month_Date"].isin(validation_months)
    ]

    X_train = train_data[["Time_Index"]]
    y_train = train_data["Quantity_Sold"]

    X_test = test_data[["Time_Index"]]
    y_test = test_data["Quantity_Sold"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    predictions = np.maximum(predictions, 0)

    for actual, predicted in zip(
        y_test,
        predictions
    ):
        validation_predictions.append(
            {
                "Product_ID": product,
                "Actual": actual,
                "Predicted": predicted
            }
        )

validation_results = pd.DataFrame(
    validation_predictions
)

# Overall metrics
mae = mean_absolute_error(
    validation_results["Actual"],
    validation_results["Predicted"]
)

rmse = np.sqrt(
    mean_squared_error(
        validation_results["Actual"],
        validation_results["Predicted"]
    )
)

# MAPE, avoiding division by zero
non_zero_actuals = (
    validation_results["Actual"] != 0
)

mape = (
    np.mean(
        np.abs(
            (
                validation_results.loc[
                    non_zero_actuals,
                    "Actual"
                ]
                -
                validation_results.loc[
                    non_zero_actuals,
                    "Predicted"
                ]
            )
            /
            validation_results.loc[
                non_zero_actuals,
                "Actual"
            ]
        )
    ) * 100
)

print("=" * 60)
print("TIME-SERIES MODEL VALIDATION")
print("=" * 60)

print("\nTraining period:")
print(training_months.min(), "to", training_months.max())

print("\nValidation period:")
print(validation_months)

print("\nValidation observations:",
      len(validation_results))

print("\nOverall MAE:",
      round(mae, 2))

print("Overall RMSE:",
      round(rmse, 2))

print("Overall MAPE:",
      round(mape, 2), "%")

print("\nValidation results:")
display(validation_results)

TIME-SERIES MODEL VALIDATION

Training period:
2024-01-01 00:00:00 to 2037-05-01 00:00:00

Validation period:
[Timestamp('2037-06-01 00:00:00'), Timestamp('2037-07-01 00:00:00'), Timestamp('2037-08-01 00:00:00')]

Validation observations: 15

Overall MAE: 407.05
Overall RMSE: 532.29
Overall MAPE: 29.99 %

Validation results:


,Product_ID,Actual,Predicted
0,P001,1496,1540.872794
1,P001,768,1540.108093
2,P001,1245,1539.343391
3,P002,1121,1697.806599
4,P002,1626,1698.473792
5,P002,1707,1699.140985
6,P003,2347,1467.445031
7,P003,926,1466.842903
8,P003,1519,1466.240775
9,P004,1276,1586.484161


In [28]:
# ==========================================
# BLOCK 24: BASELINE MODEL VALIDATION
# ==========================================

baseline_validation_predictions = []

# For each validation month, use the previous
# 3 complete months as the forecast basis
for validation_month in validation_months:

    previous_months = (
        all_months[
            all_months < validation_month
        ]
        .tolist()
    )

    previous_3_months = previous_months[-3:]

    for product in sorted(
        forecast_history["Product_ID"].unique()
    ):

        train_data = forecast_history[
            (
                forecast_history["Month_Date"]
                .isin(previous_3_months)
            )
            &
            (
                forecast_history["Product_ID"]
                == product
            )
        ]

        test_data = forecast_history[
            (
                forecast_history["Month_Date"]
                == validation_month
            )
            &
            (
                forecast_history["Product_ID"]
                == product
            )
        ]

        if len(test_data) == 0:
            continue

        actual = test_data["Quantity_Sold"].iloc[0]

        predicted = train_data[
            "Quantity_Sold"
        ].mean()

        baseline_validation_predictions.append(
            {
                "Month_Date": validation_month,
                "Product_ID": product,
                "Actual": actual,
                "Predicted": predicted
            }
        )

baseline_validation = pd.DataFrame(
    baseline_validation_predictions
)

# Metrics
baseline_mae = mean_absolute_error(
    baseline_validation["Actual"],
    baseline_validation["Predicted"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_validation["Actual"],
        baseline_validation["Predicted"]
    )
)

non_zero_baseline = (
    baseline_validation["Actual"] != 0
)

baseline_mape = (
    np.mean(
        np.abs(
            (
                baseline_validation.loc[
                    non_zero_baseline,
                    "Actual"
                ]
                -
                baseline_validation.loc[
                    non_zero_baseline,
                    "Predicted"
                ]
            )
            /
            baseline_validation.loc[
                non_zero_baseline,
                "Actual"
            ]
        )
    ) * 100
)

print("=" * 60)
print("BASELINE MODEL VALIDATION")
print("=" * 60)

print("\nValidation observations:",
      len(baseline_validation))

print("\nBaseline MAE:",
      round(baseline_mae, 2))

print("Baseline RMSE:",
      round(baseline_rmse, 2))

print("Baseline MAPE:",
      round(baseline_mape, 2), "%")

print("\nValidation results:")
display(baseline_validation)

BASELINE MODEL VALIDATION

Validation observations: 15

Baseline MAE: 539.13
Baseline RMSE: 640.01
Baseline MAPE: 38.33 %

Validation results:


,Month_Date,Product_ID,Actual,Predicted
0,2037-06-01,P001,1496,1182.000000
1,2037-06-01,P002,1121,1261.333333
2,2037-06-01,P003,2347,1093.000000
3,2037-06-01,P004,1276,2255.000000
4,2037-06-01,P005,1345,1891.333333
5,2037-07-01,P001,768,1451.666667
6,2037-07-01,P002,1626,1250.666667
7,2037-07-01,P003,926,1390.000000
8,2037-07-01,P004,1449,2027.666667
9,2037-07-01,P005,1969,1743.333333


In [30]:
# ==========================================
# BLOCK 25: FINAL NEXT-MONTH DEMAND FORECAST
# ==========================================

final_forecast_results = []

for product in sorted(
    forecast_history_model["Product_ID"].unique()
):

    product_data = forecast_history_model[
        forecast_history_model["Product_ID"] == product
    ].sort_values("Time_Index")

    X = product_data[["Time_Index"]]
    y = product_data["Quantity_Sold"]

    # Train on all complete historical months
    final_model = LinearRegression()
    final_model.fit(X, y)

    # Predict September 2037
    prediction = final_model.predict(
        pd.DataFrame(
            {"Time_Index": [next_time_index]}
        )
    )[0]

    prediction = max(0, prediction)

    final_forecast_results.append(
        {
            "Product_ID": product,
            "Forecast_Month": (
                last_complete_month
                + pd.DateOffset(months=1)
            ),
            "Forecast_Quantity": round(prediction, 2),
            "Model_Slope": round(
                final_model.coef_[0],
                4
            )
        }
    )

final_forecast = pd.DataFrame(
    final_forecast_results
)

print("=" * 60)
print("FINAL NEXT-MONTH DEMAND FORECAST")
print("=" * 60)

display(final_forecast)

print(
    "\nTotal forecasted quantity:",
    round(
        final_forecast["Forecast_Quantity"].sum(),
        2
    )
)

FINAL NEXT-MONTH DEMAND FORECAST


,Product_ID,Forecast_Month,Forecast_Quantity,Model_Slope
0,P001,2037-09-01,1511.64,-1.0094
1,P002,2037-09-01,1684.44,0.5283
2,P003,2037-09-01,1474.91,-0.5186
3,P004,2037-09-01,1606.75,0.4152
4,P005,2037-09-01,1622.70,0.8573



Total forecasted quantity: 7900.44


In [31]:
# ==========================================
# BLOCK 26: REORDER POINT CALCULATION
# ==========================================

# Calculate daily demand statistics by product
daily_demand_stats = (
    daily_demand
    .groupby("Product_ID")
    .agg(
        Avg_Daily_Demand=(
            "Quantity_Sold",
            "mean"
        ),
        Std_Daily_Demand=(
            "Quantity_Sold",
            "std"
        )
    )
    .reset_index()
)

# Get average supplier lead time by product
product_lead_time = (
    suppliers_clean
    .groupby("Product_ID")
    .agg(
        Avg_Lead_Time=(
            "Lead_Time (days)",
            "mean"
        )
    )
    .reset_index()
)

# Merge demand statistics with lead time
reorder_point = (
    daily_demand_stats
    .merge(
        product_lead_time,
        on="Product_ID",
        how="left"
    )
)

# Safety stock
reorder_point["Safety_Stock"] = (
    1.65
    * reorder_point["Std_Daily_Demand"]
    * np.sqrt(
        reorder_point["Avg_Lead_Time"]
    )
)

# Reorder Point
reorder_point["Reorder_Point"] = (
    reorder_point["Avg_Daily_Demand"]
    * reorder_point["Avg_Lead_Time"]
    + reorder_point["Safety_Stock"]
)

# Merge forecast
reorder_point = reorder_point.merge(
    final_forecast[
        [
            "Product_ID",
            "Forecast_Quantity"
        ]
    ],
    on="Product_ID",
    how="left"
)

# Round values
numeric_columns = [
    "Avg_Daily_Demand",
    "Std_Daily_Demand",
    "Avg_Lead_Time",
    "Safety_Stock",
    "Reorder_Point",
    "Forecast_Quantity"
]

reorder_point[numeric_columns] = (
    reorder_point[numeric_columns]
    .round(2)
)

print("=" * 60)
print("REORDER POINT ANALYSIS")
print("=" * 60)

display(
    reorder_point.sort_values(
        "Product_ID"
    )
)

REORDER POINT ANALYSIS


,Product_ID,Avg_Daily_Demand,Std_Daily_Demand,Avg_Lead_Time,Safety_Stock,Reorder_Point,Forecast_Quantity
0,P001,249.33,141.05,6.25,581.81,2140.10,1511.64
1,P002,264.34,144.09,6.67,613.84,2376.12,1684.44
2,P003,254.38,144.43,6.00,583.73,2110.04,1474.91
3,P004,262.57,140.66,6.00,568.50,2143.94,1606.75
4,P005,261.86,140.55,6.00,568.07,2139.25,1622.70


In [32]:
# ==========================================
# BLOCK 27: FINAL PYTHON ANALYTICS SUMMARY
# ==========================================

python_summary = final_forecast.merge(
    reorder_point[
        [
            "Product_ID",
            "Avg_Daily_Demand",
            "Avg_Lead_Time",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ],
    on="Product_ID",
    how="left"
)

python_summary = python_summary[
    [
        "Product_ID",
        "Forecast_Month",
        "Forecast_Quantity",
        "Avg_Daily_Demand",
        "Avg_Lead_Time",
        "Safety_Stock",
        "Reorder_Point",
        "Model_Slope"
    ]
]

print("=" * 70)
print("FINAL PYTHON SUPPLY CHAIN ANALYTICS SUMMARY")
print("=" * 70)

display(
    python_summary.sort_values("Product_ID")
)

print("\nTotal Forecast Quantity:",
      round(
          python_summary["Forecast_Quantity"].sum(),
          2
      ))

print("Average Reorder Point:",
      round(
          python_summary["Reorder_Point"].mean(),
          2
      ))

print("Total Reorder Point:",
      round(
          python_summary["Reorder_Point"].sum(),
          2
      ))

FINAL PYTHON SUPPLY CHAIN ANALYTICS SUMMARY


,Product_ID,Forecast_Month,Forecast_Quantity,Avg_Daily_Demand,Avg_Lead_Time,Safety_Stock,Reorder_Point,Model_Slope
0,P001,2037-09-01,1511.64,249.33,6.25,581.81,2140.10,-1.0094
1,P002,2037-09-01,1684.44,264.34,6.67,613.84,2376.12,0.5283
2,P003,2037-09-01,1474.91,254.38,6.00,583.73,2110.04,-0.5186
3,P004,2037-09-01,1606.75,262.57,6.00,568.50,2143.94,0.4152
4,P005,2037-09-01,1622.70,261.86,6.00,568.07,2139.25,0.8573



Total Forecast Quantity: 7900.44
Average Reorder Point: 2181.89
Total Reorder Point: 10909.45


In [33]:
# ==========================================
# BLOCK 28: EXPORT PROCESSED DATASETS
# ==========================================

import os

processed_path = "../data/processed"

os.makedirs(
    processed_path,
    exist_ok=True
)

# Export cleaned datasets
sales_clean.to_csv(
    f"{processed_path}/sales_clean.csv",
    index=False
)

inventory_clean.to_csv(
    f"{processed_path}/inventory_clean.csv",
    index=False
)

suppliers_clean.to_csv(
    f"{processed_path}/suppliers_clean.csv",
    index=False
)

purchase_orders_clean.to_csv(
    f"{processed_path}/purchase_orders_clean.csv",
    index=False
)

# Export analytical datasets
monthly_demand.to_csv(
    f"{processed_path}/monthly_demand.csv",
    index=False
)

product_demand_profile.to_csv(
    f"{processed_path}/product_demand_profile.csv",
    index=False
)

inventory_analysis.to_csv(
    f"{processed_path}/inventory_analysis.csv",
    index=False
)

supplier_performance.to_csv(
    f"{processed_path}/supplier_performance.csv",
    index=False
)

supplier_clustering.to_csv(
    f"{processed_path}/supplier_clustering.csv",
    index=False
)

cluster_profiles.to_csv(
    f"{processed_path}/cluster_profiles.csv",
    index=False
)

final_forecast.to_csv(
    f"{processed_path}/final_forecast.csv",
    index=False
)

reorder_point.to_csv(
    f"{processed_path}/reorder_point.csv",
    index=False
)

python_summary.to_csv(
    f"{processed_path}/python_summary.csv",
    index=False
)

print("=" * 60)
print("PROCESSED DATA EXPORT")
print("=" * 60)

for file_name in sorted(
    os.listdir(processed_path)
):
    print(file_name)

print(
    "\nTotal files exported:",
    len(os.listdir(processed_path))
)

PROCESSED DATA EXPORT
cluster_profiles.csv
final_forecast.csv
inventory_analysis.csv
inventory_clean.csv
monthly_demand.csv
product_demand_profile.csv
purchase_orders_clean.csv
python_summary.csv
reorder_point.csv
sales_clean.csv
supplier_clustering.csv
supplier_performance.csv
suppliers_clean.csv

Total files exported: 13
